# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sdukea/FAI-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring (with a ranking output)**

Lane 2 is primarily a scoring task because the objective is to estimate how worthwhile it is to investigate each webpage for refresh, optimization, or monitoring. Every page receives a Content Opportunity Score, which reflects its relative priority for human review.

The scores are then used to produce a ranked list of webpages, allowing SEO analysts, content strategists, and editors to focus their limited time on the pages with the greatest potential business impact.

Unlike a classification task, the goal is not to predict whether a page belongs to a fixed category (e.g., "good" or "bad"). Instead, the goal is to estimate the relative opportunity associated with each page and order the pages accordingly. Under the hood the score is learned the way a classifier is (from a yes/no label, see Section 2), but only the **order** it produces is used. That is why the metric in Section 3 is a ranking metric, not accuracy.

The starter dataset has **44 columns** for each of **30,000 webpages** (32 clients). Not all 44 are usable signals. The cell below sorts them by role:

- 2 IDs (grouping only)
- 2 label sources (`trend_direction`, `trend_pct`)
- 6 last-30/prev-30 window columns that rebuild the label
- 2 LLM-metadata columns the dictionary marks "not a model feature"

That leaves **32 candidate signals** describing search visibility, engagement, freshness, keyword context and content depth.

The Content Opportunity Score is estimated by combining these candidate signals to answer a business question:

> "Given limited editorial resources, which webpages should be reviewed first?"

The workflow for this lane is:

Business problem → Define what constitutes a content opportunity → Treat the 32 safe columns as candidate signals → Learn how these signals relate to content opportunity → Assign every webpage a Content Opportunity Score → Rank webpages from highest to lowest priority for human review.

The final output is not an automated content update. Instead, it is a decision-support system that recommends which pages deserve investigation first, allowing human experts to decide what improvements, if any, should be made.


In [ ]:
# Setup: works locally (Run All from work/notebooks/) and in Colab (clones this repo).
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("FAI-starter").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sdukea/FAI-starter"], check=True)
    ROOT = Path("FAI-starter")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
print(f"{len(df):,} pages x {df.shape[1]} columns | {df['client_id'].nunique()} clients")

# Not every column is a signal. Sort the 44 columns by role:
roles = {
    "id (grouping only)":                ["content_id", "client_id"],
    "label source (never a feature)":    ["trend_direction", "trend_pct"],
    "rebuilds the label (window split)": ["impressions_last_30d", "impressions_prev_30d",
                                          "clicks_last_30d", "clicks_prev_30d",
                                          "sessions_last_30d", "sessions_prev_30d"],
    "metadata, not a feature":           ["provider_used", "model_used"],
}
excluded = [c for cols in roles.values() for c in cols]
roles["candidate signal"] = [c for c in df.columns if c not in excluded]
for role, cols in roles.items():
    print(f"{len(cols):>2}  {role}")


30,000 pages x 44 columns | 32 clients
 2  id (grouping only)
 2  label source (never a feature)
 6  rebuilds the label (window split)
 2  metadata, not a feature
32  candidate signal


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For this lane, the task is not to predict whether a page is good or bad. The output is a content opportunity score for each webpage. There is no column in the dataset called `content_opportunity_score`, so the score cannot be the **target**. It is the thing the model produces. To learn it, the model needs a target that is **observed in the data**. Otherwise it would just learn whatever rule I wrote.

**My target (a proxy): `is_declining_label = (trend_direction == "down")`**

- *What it means:* the page got **more than 20% fewer** search impressions in the last 30 days than in the 30 days before (`impressions_last_30d` vs `impressions_prev_30d`).
- *Where it comes from:* an **observed outcome** (real Search Console impressions), cut into yes/no with a fixed, written threshold (−20%). The threshold is a definition, but the movement itself is measured, not decided by anyone.
- *Why it stands in for "opportunity":* a page that is losing visibility it used to have is a page an editor would want to look at. *Opportunity ≈ visibility the page is losing.*

**Why it is only a proxy (honest weaknesses):**

1. **Same window, not the future.** The label is measured in the same 90-day window as the features. So on the starter data this model *describes* current decline. It does not *predict* future decline.
2. **Noise at low volume.** A page going from 4 to 3 impressions counts as "down". The cell below measures how often that happens.
3. **Decline ≠ fixable.** A drop can be consolidation, seasonality or a SERP change (lane guide §7). The label can't tell these apart.
4. **Leakage risk.** `trend_direction`, `trend_pct` and the six last/prev-30d columns rebuild the label. They are never features (W3 data contract will audit this properly).

**The target I'm aiming for in the capstone (warehouse daily facts):**

```text
features from days [t-90, t)  ->  label: impressions in [t, t+30) fall > 20% vs [t-30, t),
                                  only for pages with a minimum prior volume
```

That is a true future outcome with no window overlap. The starter proxy is the dry run for it.

At the outset, the workflow of this lane is:

Business problem → Define what constitutes a content opportunity (the proxy above) → Treat the 32 safe columns as candidate signals → Learn how these signals relate to content opportunity → Assign every webpage a Content Opportunity Score → Rank webpages from highest to lowest priority for human review.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# proxy target:
# 1 = page is currently declining (impressions last 30d < prev 30d by more than 20%)
# 0 = page is not currently declining (stable / up / new / flat)

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# inspect target
label_counts = df['is_declining_label'].value_counts()

print("Not declining:", label_counts.get(0, 0))
print("Declining:", label_counts.get(1, 0))
print(f"Base rate: {df['is_declining_label'].mean():.1%}  <- what a random pick would score")

# check: the label really is the observed -20% impression drop (only where prev window > 0)
has_prev = df['impressions_prev_30d'] > 0
rebuilt = (df['impressions_last_30d'] < 0.8 * df['impressions_prev_30d']).astype(int)
print("Label matches the observed -20% drop:",
      f"{(rebuilt[has_prev] == df.loc[has_prev, 'is_declining_label']).mean():.1%} of pages with prev-30d impressions")

# weakness #2: how much of the label sits on tiny volumes?
tiny = df['impressions_prev_30d'] < 30
print(f"Pages with < 30 impressions in prev 30d: {tiny.sum():,} ({tiny.mean():.0%}) "
      f"— decline rate there {df.loc[tiny, 'is_declining_label'].mean():.0%}")


Not declining: 13738
Declining: 16262
Base rate: 54.2%  <- what a random pick would score
Label matches the observed -20% drop: 100.0% of pages with prev-30d impressions
Pages with < 30 impressions in prev 30d: 8,422 (28%) — decline rate there 36%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50 on held-out clients.** Of the 50 pages the score puts at the top of the queue, what share are actually declining?

**Why this one:**
- The decision is "which pages first?" with limited editor time. Only the top of the list gets reviewed, so only the top of the list should be graded. Accuracy and ROC-AUC grade the whole list, including pages nobody will ever open.
- **K = 50** is my working assumption for one review cycle's capacity (≈ a week of editor time). It's a knob, not a law. I'll also report P@20 and P@100 so the result doesn't hinge on one K.
- **Held-out clients** (whole clients kept out of training) because decline rates differ hugely by client (Section 5). Scoring on pages from clients the model has already seen would flatter it.

**What number means "good" (three bars, set *before* any training):**

| Bar | What it means | Number |
|---|---|---|
| 1. Beat random | Better than picking 50 pages blindly | P@50 > base rate (~54% overall; recompute on the held-out clients) |
| 2. Beat the obvious rule | Better than a hand-written rule an analyst would use today | P@50 > the W4 baseline rule on the *same* held-out clients |
| 3. Be worth the complexity | A clear margin, not a lucky draw | ≥ 10 points over bar 2, stable across P@20/P@100 |

**Secondary checks** (reported, not optimised): average precision (quality of the whole ranking) and the *share of the top 50 with real volume* (≥ 30 prev-30d impressions). A queue full of 4→3-impression pages could hit a high P@50 and still be useless to an editor.

Reference point (starter pipeline, client holdout, from `docs/ml-intern-dataset-and-lane-guide.md`): baseline rules P@50 = 0.24, random forest P@50 = 0.74.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

y = df['is_declining_label'].values
print(f"Bar 1 — base rate (random pick): {y.mean():.3f}\n")

# Can I compute the metric today? Yes — on the most obvious rule: "review the busiest pages first".
busiest_first = df['impressions_90d']
for k in (20, 50, 100):
    print(f"'Busiest pages first'  P@{k:<3}: {precision_at_k(busiest_first, y, k):.3f}")

# Secondary check on that same top 50: do the picks have real volume?
top50 = df.iloc[np.argsort(-busiest_first.values, kind="stable")[:50]]
print(f"\nTop-50 share with >= 30 prev-30d impressions: {(top50['impressions_prev_30d'] >= 30).mean():.0%}")


Bar 1 — base rate (random pick): 0.542

'Busiest pages first'  P@20 : 0.450
'Busiest pages first'  P@50 : 0.420
'Busiest pages first'  P@100: 0.380

Top-50 share with >= 30 prev-30d impressions: 100%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (one page) of one client, summarised over a trailing 90-day window.**

- 30,000 rows, `content_id` unique: one row per page, no duplicates to collapse.
- Pages are **nested inside 32 clients**. That's why train/test splits will be by `client_id`, never random rows.
- The ranked queue has the same grain: one row per page, with a score, a rank and (from W4) reason codes.

The cell below proves the grain and shows a few rows with the column roles from Section 1. It shows IDs (pseudonyms only), a handful of candidate signals and the label. No queries, URLs or titles exist in this slice.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# grain check: one row per page
assert df['content_id'].is_unique, "content_id is not unique — grain is not one row per page"
print(f"rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,} | clients: {df['client_id'].nunique()}")

pages_per_client = df.groupby('client_id').size()
print(f"pages per client: min {pages_per_client.min()}, median {int(pages_per_client.median())}, max {pages_per_client.max()}")

show = ['content_id', 'client_id',                                     # grouping only
        'impressions_90d', 'avg_position', 'ctr', 'engagement_rate',   # candidate signals
        'days_since_last_update', 'word_count', 'content_type',
        'is_declining_label']                                          # proxy target
df[show].sample(5, random_state=42)


rows: 30,000 | unique content_id: 30,000 | clients: 32
pages per client: min 3, median 567, max 7008


,content_id,client_id,impressions_90d,avg_position,ctr,engagement_rate,days_since_last_update,word_count,content_type,is_declining_label
2308,content_9824710082d8,client_3fdba35f04,283,21.3,0.00,0.00,104,1397.0,keyword article,0
22404,content_3efa3a7c46bb,client_f74efabef1,8878,8.8,0.09,4.35,20,3188.0,keyword article,0
23397,content_575dc8a2ab0f,client_25fc0e7096,3,0.0,0.00,0.00,8,3381.0,feedly article,0
25058,content_0dbd6911ba04,client_d029fa3a95,124,8.1,0.00,0.00,20,2892.0,comparison article,1
2664,content_bbaf87019afb,client_19581e27de,4294,30.9,0.23,2.33,22,NaN,keyword article,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

An if-statement works when one or two signals push the outcome in one consistent direction. Here they don't. The cell below measures three things:

1. **Signals interact and flip direction.** Decline rate by search position × visibility tier is not monotonic. Among top-3 pages it swings from ~11% (low visibility) to ~90% (moderate), while page-1 pages sit at 42–70%. "Good position = safe" is true in one cell and false in the next. A rule would need one hand-tuned branch per cell, and with 32 candidate signals the number of cells explodes.
2. **The obvious rule is wrong-signed.** "Review the busiest pages first" scores *below* random on P@50 (Section 3). Human intuition about which signal matters is not reliable here.
3. **Clients differ wildly.** The decline rate per client ranges from ~0% to ~94%. Any fixed threshold tuned on one client mis-fires on another. A model can learn the shape across clients, and the client-holdout test checks that it carries over to new ones.

That's the case for ML: many weak signals, tangled together, varying by client. Too messy to write by hand, but learnable from ~30k labelled pages.

**The honest counterweight:** ML has to *earn* this. The plan is a transparent rule baseline first (W4), then a model (W5), compared on the same held-out clients with the Section 3 metric. If the model doesn't clear bar 3, the rule wins because it's easier to explain to an editor.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Interactions: decline rate by position x visibility tier (cells with < 100 pages hidden as noise)
grid = df[df['position_tier'] != 'no_data']
rate = pd.crosstab(grid['position_tier'], grid['impression_tier'], values=grid['is_declining_label'], aggfunc='mean')
n    = pd.crosstab(grid['position_tier'], grid['impression_tier'])
rows = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
cols = ['low', 'moderate', 'good', 'excellent']
print("Decline rate by position tier x impression tier (n >= 100):")
print(rate.where(n >= 100).reindex(index=rows, columns=cols).round(2).to_string(na_rep='  -'))

# 2. The obvious rule vs random (from Section 3)
print(f"\n'Busiest pages first' P@50: {precision_at_k(df['impressions_90d'], y, 50):.2f}  vs random {y.mean():.2f}")

# 3. Clients differ wildly
by_client = df.groupby('client_id')['is_declining_label'].mean()
print(f"\nDecline rate per client: min {by_client.min():.0%}, median {by_client.median():.0%}, max {by_client.max():.0%}")


Decline rate by position tier x impression tier (n >= 100):
impression_tier   low  moderate  good  excellent
position_tier                                   
top_3            0.11      0.90  0.70          -
page_1           0.53      0.70  0.53       0.42
striking         0.59      0.63  0.60          -
page_3_5         0.48      0.54  0.69       0.61
deep             0.38      0.28     -          -

'Busiest pages first' P@50: 0.42  vs random 0.54

Decline rate per client: min 0%, median 52%, max 94%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.